In [ ]:
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from mask_dataset import WheatBinaryDataset

In [ ]:
DEVICE = "cuda"
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

# Training Config
train_transform = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])


In [ ]:
train_dataset = WheatBinaryDataset('masks/train', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

val_dataset = WheatBinaryDataset('masks/valid', transform=train_transform) # or your val_transform
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

test_dataset = WheatBinaryDataset('masks/test', transform=train_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)



In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1, # Binary segmentation
).to(DEVICE)


In [ ]:
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    metric_iou = 0
    metric_f1 = 0

    # We use a threshold of 0.5 to turn the model's Sigmoid output into a binary 0/1 mask
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)

        # Forward pass
        output = model(images)

        # Apply sigmoid and threshold
        tp, fp, fn, tn = smp.metrics.get_stats(
            output,
            masks.long(),
            mode='binary',
            threshold=0.5
        )

        # Calculate IoU and F1 (Dice) for this batch
        metric_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
        metric_f1 += smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")

    avg_iou = metric_iou / len(loader)
    avg_f1 = metric_f1 / len(loader)

    return avg_iou, avg_f1

In [ ]:

best_val_loss = float('inf')


for epoch in range(EPOCHS):

    print(f"\n--- Starting Epoch {epoch+1} ---", flush=True)

    model.train()
    train_loss = 0
    for batch_idx, (images, masks) in enumerate(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        # Print every 5 batches to show signs of life
        if batch_idx % 5 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}", flush=True)

    model.eval()
    val_loss = 0
    metric_iou = 0
    with torch.no_grad(): # Disable gradient tracking to save memory
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            output = model(images)

            v_loss = criterion(output, masks)
            val_loss += v_loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    # Update the Learning Rate based on Validation Loss
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'low_loss_canopy_seg_model.pth')
        print(f"Epoch {epoch+1}: Val Loss Improved to {avg_val_loss:.4f}. Model Saved!", flush=True)
    else:
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}", flush=True)

In [ ]:


# Evaluation
val_iou, val_f1 = evaluate_model(model, val_loader, DEVICE)
test_iou, test_f1 = evaluate_model(model, test_loader, DEVICE)

print(f"Validation mIoU: {val_iou:.4f} | Validation F1 (Dice): {val_f1:.4f}")
print(f"Test mIoU: {test_iou:.4f} | Test F1 (Dice): {test_f1:.4f}")

In [ ]:
torch.save(model.state_dict(), 'best_canopy_seg_model.pth')

print("Model weights saved successfully.")